# Carte mel et autoencodeur : préparer une entrée, puis la reconstruire

Faire une seule préparation explicite et suivre la carte jusqu’à sa reconstruction.

Nous lisons et exécutons les étapes dans l’ordre. Les cellules « Observer » montrent les valeurs et les formes ; elles ne prouvent pas la réussite des exercices.

Environnement vérifié : Python 3.11.9, PyTorch 2.10.0 CPU et TorchAudio 2.10.0 lorsque le chapitre le demande. Aucun modèle ni jeu de données n’est téléchargé pendant l’exécution.

[Revenir au chapitre](https://radiocodeur.github.io/quiz/cours.html?theme=pytorch#conv-mel-autoencodeur).


## Préparer le noyau

Choisissez un environnement CPU neuf. En cas de bibliothèque manquante, installez-la dans ce noyau, puis redémarrez-le. Les sorties affichent les versions réellement utilisées.

In [1]:
import os
import sys
import tempfile

# Choisir un noyau neuf pour garantir la vérification sur CPU.
os.environ["CUDA_VISIBLE_DEVICES"] = ""
try:
    import torch
except ImportError as error:
    raise RuntimeError("PyTorch manque. Installer torch==2.10.0 dans ce noyau, puis redémarrer et relancer.") from error
torch.set_num_threads(1)
torch.manual_seed(17)
print("Python", sys.version.split()[0], "— PyTorch", torch.__version__, "— CPU")
assert not torch.cuda.is_available(), "Redémarrer dans un noyau CPU neuf."
_quiz_initial_directory = os.getcwd()
_quiz_temporary_directory = tempfile.TemporaryDirectory(prefix="quiz-progressif-")
os.chdir(_quiz_temporary_directory.name)


Python 3.11.9 — PyTorch 2.10.0+cpu — CPU


In [2]:
try:
    import torchaudio
except (ImportError, OSError) as error:
    raise RuntimeError("TorchAudio manque ou ne correspond pas à PyTorch. Installer torch==2.10.0 et torchaudio==2.10.0 ensemble, puis redémarrer.") from error
print("TorchAudio",torchaudio.__version__)

TorchAudio 2.10.0+cpu


In [3]:
# Cet outil sert uniquement à afficher les états des démonstrations.
# Il ne modifie pas les tenseurs et ne fait pas partie du modèle étudié.
def observer(nom, valeur):
    if isinstance(valeur, torch.Tensor):
        contenu = valeur.detach().cpu()
        print(nom, "forme", tuple(contenu.shape), "type", contenu.dtype,
              "requires_grad", valeur.requires_grad)
        if contenu.numel() <= 16:
            print("  valeurs :", contenu.tolist())
    elif valeur is None or isinstance(valeur, (int, float, bool, str, tuple)):
        print(nom, "=", valeur)
    elif isinstance(valeur, list) and len(valeur) <= 8:
        if all(isinstance(item,(int,float,bool,str)) for item in valeur):
            print(nom, "=", valeur)
        else:
            print(nom, ":",len(valeur),"éléments ; formes des tenseurs :",
                  [tuple(item.shape) for item in valeur if isinstance(item,torch.Tensor)])
    elif isinstance(valeur,dict):
        print(nom, ": clés",list(valeur)[:12])

def observer_entrainement(espace, montrer_poids=False):
    # Une copie évite de parcourir un dictionnaire global en mutation.
    vus = set()
    for nom, objet in list(espace.items()):
        if nom.startswith('_'):
            continue
        if isinstance(objet, torch.Tensor) and objet.is_leaf and objet.requires_grad:
            parametres = [(nom, objet)]
        elif isinstance(objet, torch.nn.Module):
            parametres = [(nom + '.' + cle, parametre)
                          for cle, parametre in list(objet.named_parameters())[:4]]
        else:
            continue
        for cle, parametre in parametres:
            if id(parametre) in vus:
                continue
            vus.add(id(parametre))
            if montrer_poids:
                observer(cle, parametre)
            observer(cle + '.grad', parametre.grad)


Le filtre sait parcourir une grille ; donnons-lui une représentation issue d’un signal. Nous changeons de données : deux sinusoïdes synthétiques de 220 et 440 Hz, une seconde à 16 kHz, amplitude 0,2. Elles isolent les formes sans fichier à télécharger. Ce ne sont pas des voix enregistrées.

Une transformation mel calcule des puissances par trame puis les regroupe en bandes. Nous choisissons une fenêtre de Hann de 400 valeurs, un pas de 160, 64 bandes et aucun centrage ajouté. Il reste 98 trames : (16000−400)/160, partie entière, puis +1.

| Objet | Axes et forme | Pourquoi |
| --- | --- | --- |
| Ondes | (2,16000) | Deux signaux d’une seconde |
| Mel | (2,64,98) | 64 bandes, 98 trames |
| Entrée réseau | (2,1,64,96) | Un canal ; deux réductions par deux prévues |

Les conventions mel Slaney et les détails de STFT sont repris du module Signal.

La division par la somme de fenêtre au carré fixe notre normalisation de puissance. Le plancher 10⁻¹⁰ évite log(0) ; 10×log10 donne des dB par rapport à la référence numérique 1. Diviser par 100 réduit leur échelle pour ce réseau. Ce ne sont ni des probabilités ni des dB SPL calibrés.

Nous gardons 96 trames pour que deux réductions par deux tombent juste, puis ajoutons un canal de caractéristiques. Cette entrée est maintenant prête : nous la réutilisons, sans recalculer la préparation à chaque couche.

Sequential enchaîne les couches. Nous intercalons une ReLU : elle garde les nombres positifs et remplace les négatifs par zéro. Ainsi [−2,0,3] donne [0,0,3], sans changer de forme. nn.ReLU() prépare cette opération pour l’enchaînement ; torch.relu(x) l’applique directement à x. Les convolutions de stride 2 réduiront les bandes et le temps ; les canaux 4 et 8 seront des caractéristiques apprises.

Le premier bloc produit (2,4,32,48), le second (2,8,16,24). Nous avons comprimé les deux axes de la carte. Il reste à réagrandir cette représentation pour la comparer à notre cible.

| Étape | Forme : extraits, canaux, bandes, trames |
| --- | --- |
| Entrée | (2,1,64,96) |
| Première convolution | (2,4,32,48) |
| Seconde convolution, latent | (2,8,16,24) |
| Première transposée | (2,4,32,48) |
| Seconde transposée | (2,1,64,96) |

Le décodeur utilise des convolutions transposées pour apprendre le réagrandissement. Le nombre d’extraits reste deux.

Le décodeur retrouve (2,1,64,96), mais cette compatibilité de forme n’inverse pas les valeurs perdues. Sa dernière couche reste sans ReLU : la cible logarithmique possède des valeurs négatives. Une convolution transposée apprend un réagrandissement, pas un inverse garanti.

La cible est entree elle-même, pas une classe. MSELoss prend la moyenne des carrés des écarts entre ses valeurs et celles de la reconstruction. Les gradients indiquent alors comment modifier les paramètres pour diminuer cette erreur.

Avec le départ reproductible du programme, un pas fait passer la MSE d’environ 0,876 à 0,797. Cela prouve seulement que cette erreur numérique baisse sur deux cartes. Le décodeur ne produit pas d’onde : phase, écoute et généralisation restent à traiter séparément.

## Démonstration 1 · Une préparation mel, un encodeur et un pas de reconstruction

Retrouver les objets et exécuter les fragments du chapitre dans leur contexte complet.

### Préparer les bibliothèques

La graine fixe le départ des couches ; les deux ondes sont déterministes et synthétiques.

In [4]:
import torch
import torchaudio
from torch import nn


### Calculer l’état suivant

Exécutez cette petite partie après la précédente. Si un objet est créé, la cellule d’observation suivante permet d’en examiner l’état.

In [5]:
torch.manual_seed(17)
sr = 16000
temps = torch.arange(sr) / sr
onde_220 = .2 * torch.sin(2 * torch.pi * 220 * temps)


In [6]:
observer('sr', sr)
observer('temps', temps)
observer('onde_220', onde_220)

sr = 16000
temps forme (16000,) type torch.float32 requires_grad False
onde_220 forme (16000,) type torch.float32 requires_grad False


### Calculer l’état suivant

Exécutez cette petite partie après la précédente. Si un objet est créé, la cellule d’observation suivante permet d’en examiner l’état.

In [7]:
onde_440 = .2 * torch.sin(2 * torch.pi * 440 * temps)
ondes = torch.stack([onde_220, onde_440])


In [8]:
observer('onde_440', onde_440)
observer('ondes', ondes)

onde_440 forme (16000,) type torch.float32 requires_grad False
ondes forme (2, 16000) type torch.float32 requires_grad False


### Calculer l’état suivant

Hann et les paramètres de trame déterminent 64 bandes et 98 trames.

In [9]:
fenetre = torch.hann_window(400)
transformation = torchaudio.transforms.MelSpectrogram(
    sample_rate=sr, n_fft=400, win_length=400, hop_length=160,
    n_mels=64, f_min=0., f_max=8000., center=False,
    mel_scale="slaney", norm="slaney", power=2.)
mel_brut = transformation(ondes)
puissance = mel_brut / fenetre.sum().square()


In [10]:
observer('fenetre', fenetre)
observer('transformation', transformation)
observer('mel_brut', mel_brut)
observer('puissance', puissance)

fenetre forme (400,) type torch.float32 requires_grad False
mel_brut forme (2, 64, 98) type torch.float32 requires_grad False
puissance forme (2, 64, 98) type torch.float32 requires_grad False


### Calculer l’état suivant

Le plancher protège le logarithme ; la découpe et l’axe canal préparent les convolutions.

In [11]:
mel_db = 10 * puissance.clamp_min(1e-10).log10()
mel_reduit = mel_db / 100
mel_court = mel_reduit[:, :, :96]
entree = mel_court.unsqueeze(1)


In [12]:
observer('mel_db', mel_db)
observer('mel_reduit', mel_reduit)
observer('mel_court', mel_court)
observer('entree', entree)

mel_db forme (2, 64, 98) type torch.float32 requires_grad False
mel_reduit forme (2, 64, 98) type torch.float32 requires_grad False
mel_court forme (2, 64, 96) type torch.float32 requires_grad False
entree forme (2, 1, 64, 96) type torch.float32 requires_grad False


### Calculer l’état suivant

ReLU applique max(0,x). L’encodeur réduit deux fois ; le décodeur réagrandit deux fois, sans ReLU finale.

In [13]:
activation_exemple = nn.ReLU()(torch.tensor([-2., 0., 3.]))
encodeur = nn.Sequential(
    nn.Conv2d(1, 4, 3, stride=2, padding=1), nn.ReLU(),
    nn.Conv2d(4, 8, 3, stride=2, padding=1), nn.ReLU())
decodeur = nn.Sequential(
    nn.ConvTranspose2d(8, 4, 4, stride=2, padding=1), nn.ReLU(),
    nn.ConvTranspose2d(4, 1, 4, stride=2, padding=1))


In [14]:
observer('activation_exemple', activation_exemple)

activation_exemple forme (3,) type torch.float32 requires_grad False
  valeurs : [0.0, 0.0, 3.0]


### Calculer l’état suivant

Exécutez cette petite partie après la précédente. Si un objet est créé, la cellule d’observation suivante permet d’en examiner l’état.

In [15]:
modele = nn.Sequential(encodeur, decodeur)


### Choisir les paramètres et leur mise à jour

La cible et la reconstruction partagent les mêmes axes. Adam est préparé sans avoir encore modifié les poids.

In [16]:
latent = encodeur(entree)
reconstruction = decodeur(latent)
critere = nn.MSELoss()
optimiseur = torch.optim.Adam(modele.parameters(), lr=.01)


In [17]:
observer('latent', latent)
observer('reconstruction', reconstruction)

latent forme (2, 8, 16, 24) type torch.float32 requires_grad True
reconstruction forme (2, 1, 64, 96) type torch.float32 requires_grad True


### Remettre les gradients à zéro

Exécutez cette petite partie après la précédente. Si un objet est créé, la cellule d’observation suivante permet d’en examiner l’état.

In [18]:
optimiseur.zero_grad()


In [19]:
observer_entrainement({'sr': sr, 'temps': temps, 'onde_220': onde_220, 'onde_440': onde_440, 'ondes': ondes, 'fenetre': fenetre, 'transformation': transformation, 'mel_brut': mel_brut, 'puissance': puissance, 'mel_db': mel_db, 'mel_reduit': mel_reduit, 'mel_court': mel_court, 'entree': entree, 'activation_exemple': activation_exemple, 'encodeur': encodeur, 'decodeur': decodeur, 'modele': modele, 'latent': latent, 'reconstruction': reconstruction, 'critere': critere, 'optimiseur': optimiseur}, montrer_poids=False)

encodeur.0.weight.grad = None
encodeur.0.bias.grad = None
encodeur.2.weight.grad = None
encodeur.2.bias.grad = None
decodeur.0.weight.grad = None
decodeur.0.bias.grad = None
decodeur.2.weight.grad = None
decodeur.2.bias.grad = None


### Calculer l’état suivant

Exécutez cette petite partie après la précédente. Si un objet est créé, la cellule d’observation suivante permet d’en examiner l’état.

In [20]:
perte = critere(reconstruction, entree)
avant = perte.item()


In [21]:
observer('perte', perte)
observer('avant', avant)

perte forme () type torch.float32 requires_grad True
  valeurs : 0.8759986758232117
avant = 0.8759986758232117


### Calculer les gradients

backward calcule les gradients ; step change les poids. La seconde mesure est nouvelle et sans graphe.

In [22]:
perte.backward()


In [23]:
observer_entrainement({'sr': sr, 'temps': temps, 'onde_220': onde_220, 'onde_440': onde_440, 'ondes': ondes, 'fenetre': fenetre, 'transformation': transformation, 'mel_brut': mel_brut, 'puissance': puissance, 'mel_db': mel_db, 'mel_reduit': mel_reduit, 'mel_court': mel_court, 'entree': entree, 'activation_exemple': activation_exemple, 'encodeur': encodeur, 'decodeur': decodeur, 'modele': modele, 'latent': latent, 'reconstruction': reconstruction, 'critere': critere, 'optimiseur': optimiseur, 'perte': perte, 'avant': avant}, montrer_poids=False)

encodeur.0.weight.grad forme (4, 1, 3, 3) type torch.float32 requires_grad False
encodeur.0.bias.grad forme (4,) type torch.float32 requires_grad False
  valeurs : [-0.0020042110700160265, 0.013977506197988987, 0.023997362703084946, 0.00016853325359988958]
encodeur.2.weight.grad forme (8, 4, 3, 3) type torch.float32 requires_grad False
encodeur.2.bias.grad forme (8,) type torch.float32 requires_grad False
  valeurs : [0.0002566033217590302, -0.0025012814439833164, 0.0, 0.0, 0.007029403932392597, 0.0, 0.02288668043911457, 0.01873181201517582]
decodeur.0.weight.grad forme (8, 4, 4, 4) type torch.float32 requires_grad False
decodeur.0.bias.grad forme (4,) type torch.float32 requires_grad False
  valeurs : [-0.14298169314861298, 0.13798876106739044, 0.035217031836509705, 0.053063664585351944]
decodeur.2.weight.grad forme (4, 1, 4, 4) type torch.float32 requires_grad False
decodeur.2.bias.grad forme (1,) type torch.float32 requires_grad False
  valeurs : [1.845211386680603]


### Appliquer une mise à jour

Exécutez cette petite partie après la précédente. Si un objet est créé, la cellule d’observation suivante permet d’en examiner l’état.

In [24]:
optimiseur.step()


In [25]:
observer_entrainement({'sr': sr, 'temps': temps, 'onde_220': onde_220, 'onde_440': onde_440, 'ondes': ondes, 'fenetre': fenetre, 'transformation': transformation, 'mel_brut': mel_brut, 'puissance': puissance, 'mel_db': mel_db, 'mel_reduit': mel_reduit, 'mel_court': mel_court, 'entree': entree, 'activation_exemple': activation_exemple, 'encodeur': encodeur, 'decodeur': decodeur, 'modele': modele, 'latent': latent, 'reconstruction': reconstruction, 'critere': critere, 'optimiseur': optimiseur, 'perte': perte, 'avant': avant}, montrer_poids=True)

encodeur.0.weight forme (4, 1, 3, 3) type torch.float32 requires_grad True
encodeur.0.weight.grad forme (4, 1, 3, 3) type torch.float32 requires_grad False
encodeur.0.bias forme (4,) type torch.float32 requires_grad True
  valeurs : [0.19888471066951752, 0.14376017451286316, -0.046687014400959015, -0.15152062475681305]
encodeur.0.bias.grad forme (4,) type torch.float32 requires_grad False
  valeurs : [-0.0020042110700160265, 0.013977506197988987, 0.023997362703084946, 0.00016853325359988958]
encodeur.2.weight forme (8, 4, 3, 3) type torch.float32 requires_grad True
encodeur.2.weight.grad forme (8, 4, 3, 3) type torch.float32 requires_grad False
encodeur.2.bias forme (8,) type torch.float32 requires_grad True
  valeurs : [-0.16481776535511017, 0.006333737634122372, 0.09473259747028351, -0.15995699167251587, 0.031158989295363426, -0.07432162761688232, -0.06967157870531082, 0.01636720821261406]
encodeur.2.bias.grad forme (8,) type torch.float32 requires_grad False
  valeurs : [0.000256603

### Appliquer le contexte explicite

Exécutez cette petite partie après la précédente. Si un objet est créé, la cellule d’observation suivante permet d’en examiner l’état.

In [26]:
with torch.no_grad():
    apres = critere(modele(entree), entree).item()


### Lire les sorties

Le bilan concerne des cartes mel synthétiques, pas une onde ou une qualité d’écoute.

In [27]:
print("ondes :", tuple(ondes.shape))
print("mel avant decoupe :", tuple(mel_db.shape))
print("entree / latent / reconstruction :", tuple(entree.shape), tuple(latent.shape), tuple(reconstruction.shape))
print(f"MSE avant={avant:.6f}, apres={apres:.6f}")


ondes : (2, 16000)
mel avant decoupe : (2, 64, 98)
entree / latent / reconstruction : (2, 1, 64, 96) (2, 8, 16, 24) (2, 1, 64, 96)
MSE avant=0.875999, apres=0.796794


In [28]:
assert bool(tuple(ondes.shape) == (2,16000) and tuple(mel_db.shape) == (2,64,98)), 'conv-mel-autoencodeur/main: tuple(ondes.shape) == (2,16000) and tuple(mel_db.shape) == (2,64,98)'
assert bool(tuple(entree.shape) == (2,1,64,96) and tuple(latent.shape) == (2,8,16,24)), 'conv-mel-autoencodeur/main: tuple(entree.shape) == (2,1,64,96) and tuple(latent.shape) == (2,8,16,24)'
assert bool(reconstruction.shape == entree.shape and (entree < 0).any().item()), 'conv-mel-autoencodeur/main: reconstruction.shape == entree.shape and (entree < 0).any().item()'
assert bool(activation_exemple.tolist() == [0.,0.,3.]), 'conv-mel-autoencodeur/main: activation_exemple.tolist() == [0.,0.,3.]'
assert bool(abs(avant-.8759986758232117) < 2e-6 and abs(apres-.796794) < 2e-6 and apres < avant), 'conv-mel-autoencodeur/main: abs(avant-.8759986758232117) < 2e-6 and abs(apres-.796794) < 2e-6 and apres < avant'
print("Contrôles numériques de la démonstration réussis.")

Contrôles numériques de la démonstration réussis.


## À vous de modifier le calcul

Les deux cellules suivantes sont prêtes à éditer. Si vous les exécutez sans les compléter, elles le signalent et ne prétendent pas que vous avez réussi. Une réponse fausse déclenche le contrôle correspondant.

### Exercice 1

Conv2d(1,4,3,stride=2,padding=1) reçoit une carte préparée (2,1,64,96). Quelle forme produit-elle ?

In [29]:
exo_x=torch.zeros(2,1,64,96)
exo_conv=torch.nn.Conv2d(1,4,3,stride=2,padding=1)
exo_forme = ...  # Remplacer les points par votre réponse.
if exo_forme is Ellipsis or exo_forme is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    assert tuple(exo_forme) == tuple(exo_conv(exo_x).shape)
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")


À compléter : cet exercice n’est pas encore réalisé.


<details>
<summary>Indice pour l’exercice 1</summary>

Le lot reste deux, les canaux deviennent quatre et les deux axes spatiaux sont divisés par deux ici.

</details>

### Exercice 2

Une cible logarithmique contient les valeurs choisies [−0,8 ; −0,2]. Quelle MSE minimale donne une sortie forcée à zéro par ReLU ?

In [30]:
exo_cible=torch.tensor([-.8,-.2])
exo_sortie=torch.nn.ReLU()(exo_cible)
exo_perte = ...  # Remplacer les points par votre réponse.
if exo_perte is Ellipsis or exo_perte is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    torch.testing.assert_close(torch.as_tensor(exo_perte),torch.nn.functional.mse_loss(exo_sortie,exo_cible))
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")


À compléter : cet exercice n’est pas encore réalisé.


<details>
<summary>Indice pour l’exercice 2</summary>

Le meilleur nombre autorisé par ReLU pour chaque cible négative est zéro. Faites la moyenne de 0,8² et 0,2².

</details>

## Comparer avec les solutions

Ouvrez-les après avoir formulé votre réponse et votre explication. Les solutions sont également exécutées par le vérificateur du dépôt.

<details>
<summary>Solution de l’exercice 1</summary>

```python
exo_x=torch.zeros(2,1,64,96)
exo_conv=torch.nn.Conv2d(1,4,3,stride=2,padding=1)
exo_forme = (2,4,32,48)
if exo_forme is Ellipsis or exo_forme is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    assert tuple(exo_forme) == tuple(exo_conv(exo_x).shape)
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")
```

</details>

<details>
<summary>Solution de l’exercice 2</summary>

```python
exo_cible=torch.tensor([-.8,-.2])
exo_sortie=torch.nn.ReLU()(exo_cible)
exo_perte = 0.34
if exo_perte is Ellipsis or exo_perte is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    torch.testing.assert_close(torch.as_tensor(exo_perte),torch.nn.functional.mse_loss(exo_sortie,exo_cible))
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")
```

</details>

In [33]:
os.chdir(_quiz_initial_directory)
_quiz_temporary_directory.cleanup()